# Final Evaluation

Evaluates one **manually frozen configuration** on the untouched **9 final-evaluation images**.

No recommendation or retuning is performed here.
Before running the evaluation cell, enter your final values below and change `FINAL_SETTINGS_CONFIRMED = True`.


In [1]:
from pathlib import Path
from dataclasses import replace
from itertools import product
import random
import sys

import cv2
import numpy as np
import pandas as pd


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in [start, *start.parents]:
        if (
            (candidate / "banana_quality" / "config.py").exists()
            and (candidate / "data" / "annotations").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not find project root. Start Jupyter from inside the banana project."
    )

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from banana_quality.config import ProcessingConfig
from banana_quality.segmentation.kmeans import prepare_image
from banana_quality.segmentation.mask_refinement import banana_mask_from_prepared
from banana_quality.blemish.detection import blemish_mask_from_prepared
from banana_quality.surface_quality import blemish_percentage
from banana_quality.morphology import _morphology, _remove_tiny_components

DATASET_ROOT = PROJECT_ROOT / "data" / "annotations"
RESULTS_DIR = PROJECT_ROOT / "evaluation" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
SPLIT_SEED = 42
FINAL_EVALUATION_COUNT = 9

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)


Project root: c:\Users\enton\Project\python\banana_quality_ripeness_assessment_system
Dataset root: c:\Users\enton\Project\python\banana_quality_ripeness_assessment_system\data\annotations


In [2]:
images_dir = DATASET_ROOT / "images"
image_paths = sorted(
    p for p in images_dir.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

shuffled_paths = image_paths.copy()
rng = random.Random(SPLIT_SEED)
rng.shuffle(shuffled_paths)
final_names = {p.name for p in shuffled_paths[:FINAL_EVALUATION_COUNT]}

split_df = pd.DataFrame([
    {
        "image": p.name,
        "split": "final_evaluation" if p.name in final_names else "tuning",
    }
    for p in image_paths
]).sort_values(["split", "image"]).reset_index(drop=True)

split_df.to_csv(RESULTS_DIR / "dataset_split.csv", index=False)

assert len(image_paths) == 30, f"Expected 30 images, found {len(image_paths)}"
assert int((split_df["split"] == "tuning").sum()) == 21
assert int((split_df["split"] == "final_evaluation").sum()) == 9

print(split_df["split"].value_counts())


split
tuning              21
final_evaluation     9
Name: count, dtype: int64


In [3]:
def matching_mask(folder: Path, image_path: Path) -> Path:
    path = folder / f"{image_path.stem}.png"
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def load_binary_mask(path: Path, target_shape: tuple[int, int]) -> np.ndarray:
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f"Cannot read mask: {path}")
    mask = np.where(mask > 0, 255, 0).astype(np.uint8)
    if mask.shape != target_shape:
        h, w = target_shape
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
    return mask


def resize_binary_mask(mask: np.ndarray, target_shape: tuple[int, int]) -> np.ndarray:
    h, w = target_shape
    resized = cv2.resize(
        np.asarray(mask, dtype=np.uint8),
        (w, h),
        interpolation=cv2.INTER_NEAREST,
    )
    return np.where(resized > 0, 255, 0).astype(np.uint8)


def iou_score(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    pred = prediction > 0
    gt = ground_truth > 0
    union = int(np.logical_or(pred, gt).sum())
    if union == 0:
        return 1.0
    return float(np.logical_and(pred, gt).sum() / union)


def dice_score(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    pred = prediction > 0
    gt = ground_truth > 0
    denominator = int(pred.sum() + gt.sum())
    if denominator == 0:
        return 1.0
    return float(2.0 * np.logical_and(pred, gt).sum() / denominator)


def load_samples(split_name: str):
    names = set(split_df.loc[split_df["split"] == split_name, "image"])
    samples = []
    for image_path in image_paths:
        if image_path.name not in names:
            continue
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            raise ValueError(f"Cannot read image: {image_path}")
        shape = image.shape[:2]
        banana_gt = load_binary_mask(
            matching_mask(DATASET_ROOT / "banana_masks", image_path),
            shape,
        )
        blemish_gt = load_binary_mask(
            matching_mask(DATASET_ROOT / "blemish_masks", image_path),
            shape,
        )
        samples.append({
            "name": image_path.name,
            "image": image,
            "shape": shape,
            "banana_gt": banana_gt,
            "blemish_gt": blemish_gt,
        })
    return samples


def clean_variant(raw_mask, banana_mask, opening, closing, min_area_ratio):
    raw = np.where(np.asarray(raw_mask) > 0, 255, 0).astype(np.uint8)
    opened = _morphology(raw, cv2.MORPH_OPEN, int(opening))
    closed = _morphology(opened, cv2.MORPH_CLOSE, int(closing))
    minimum_area = max(
        8,
        int((np.asarray(banana_mask) > 0).sum() * float(min_area_ratio)),
    )
    final_mask = _remove_tiny_components(closed, minimum_area)
    final_mask[np.asarray(banana_mask) == 0] = 0
    return final_mask


def variant_mask_from_steps(
    steps,
    banana_mask,
    use_rule_a,
    use_rule_b,
    opening,
    closing,
    min_area_ratio,
):
    candidate = np.asarray(steps["kmeans_region_candidate"]) > 0
    if use_rule_a:
        candidate |= np.asarray(steps["rule_a_mask"]) > 0
    if use_rule_b:
        candidate |= np.asarray(steps["rule_b_mask"]) > 0
    return clean_variant(
        np.where(candidate, 255, 0).astype(np.uint8),
        banana_mask,
        opening,
        closing,
        min_area_ratio,
    )


## 1. Load tuned core parameters and manually enter the final rule settings

The core K-Means/morphology values come automatically from Notebook 01. Rule choice and rule thresholds remain manual because they should be decided after ablation + sensitivity analysis.


In [5]:
core_path = RESULTS_DIR / "recommended_core_parameters.csv"
if not core_path.exists():
    raise FileNotFoundError(
        "Run 01_parameter_tuning.ipynb first. Missing: " + str(core_path)
    )

core = pd.read_csv(core_path).iloc[0]

BANANA_K = int(core["kmeans_k"])
BANANA_ACCEPTANCE = float(core["banana_cluster_score_threshold"])
BANANA_OPENING = int(core["banana_opening_kernel"])
BANANA_CLOSING = int(core["banana_closing_kernel"])
BLEMISH_K = int(core["blemish_kmeans_k"])
BLEMISH_CHROMA = float(core["blemish_cluster_chroma_threshold"])
BLEMISH_OPENING = int(core["blemish_opening_kernel"])
BLEMISH_CLOSING = int(core["blemish_closing_kernel"])

print("Loaded tuned core parameters from Notebook 01:")
display(pd.DataFrame([{
    "banana_k": BANANA_K,
    "banana_acceptance": BANANA_ACCEPTANCE,
    "banana_opening": BANANA_OPENING,
    "banana_closing": BANANA_CLOSING,
    "blemish_k": BLEMISH_K,
    "blemish_chroma": BLEMISH_CHROMA,
    "blemish_opening": BLEMISH_OPENING,
    "blemish_closing": BLEMISH_CLOSING,
}]))

# Rule choice — decide from Notebook 02.
USE_RULE_A = True
USE_RULE_B = True

# Final rule thresholds — decide yourself after reviewing Notebook 03.
HUE_DIFFERENCE = 7.0
DARKNESS = 36.0
STRONG_HUE_DIFFERENCE = 10.0
REDNESS = 11.0
YELLOW_LOSS = 20.0
MIN_SATURATION = 35.0
HIGHLIGHT_VALUE = 220.0
HIGHLIGHT_SATURATION = 100.0

# Safety switch: set True only after you have manually frozen the rule settings above.
FINAL_SETTINGS_CONFIRMED = True

if not FINAL_SETTINGS_CONFIRMED:
    raise RuntimeError(
        "Review the ablation and sensitivity results, freeze the final rule settings, "
        "then set FINAL_SETTINGS_CONFIRMED = True."
    )

DEFAULTS = ProcessingConfig()
final_config = replace(
    DEFAULTS,
    kmeans_k=int(BANANA_K),
    banana_cluster_score_threshold=float(BANANA_ACCEPTANCE),
    blemish_kmeans_k=int(BLEMISH_K),
    blemish_cluster_chroma_threshold=float(BLEMISH_CHROMA),
    hue_drop_threshold=float(HUE_DIFFERENCE),
    darkness_threshold=float(DARKNESS),
    strong_hue_drop_threshold=float(STRONG_HUE_DIFFERENCE),
    redness_threshold=float(REDNESS),
    yellow_loss_threshold=float(YELLOW_LOSS),
    blemish_min_saturation=float(MIN_SATURATION),
    highlight_value_threshold=float(HIGHLIGHT_VALUE),
    highlight_saturation_threshold=float(HIGHLIGHT_SATURATION),
)


Loaded tuned core parameters from Notebook 01:


,banana_k,banana_acceptance,banana_opening,banana_closing,blemish_k,blemish_chroma,blemish_opening,blemish_closing
0,7,0.35,3,7,6,21.0,3,3


## 2. Evaluate only the 9 untouched images


In [6]:
final_samples = load_samples("final_evaluation")
assert len(final_samples) == 9
rows = []
for sample in final_samples:
    prepared = prepare_image(sample["image"], final_config)
    banana_steps = banana_mask_from_prepared(
        prepared,
        opening_kernel_size=int(BANANA_OPENING),
        closing_kernel_size=int(BANANA_CLOSING),
    )
    banana_working = np.asarray(banana_steps["banana_mask"], dtype=np.uint8)
    banana_pred = resize_binary_mask(banana_working, sample["shape"])

    blemish_steps = blemish_mask_from_prepared(
        prepared,
        banana_working,
        opening_kernel_size=int(BLEMISH_OPENING),
        closing_kernel_size=int(BLEMISH_CLOSING),
        config=final_config,
    )
    blemish_working = variant_mask_from_steps(
        blemish_steps,
        banana_working,
        use_rule_a=USE_RULE_A,
        use_rule_b=USE_RULE_B,
        opening=BLEMISH_OPENING,
        closing=BLEMISH_CLOSING,
        min_area_ratio=final_config.min_blemish_area_ratio,
    )
    blemish_pred = resize_binary_mask(blemish_working, sample["shape"])

    gt_pct = blemish_percentage(sample["banana_gt"], sample["blemish_gt"])
    pred_pct = blemish_percentage(banana_pred, blemish_pred)
    rows.append({
        "image": sample["name"],
        "banana_iou": iou_score(banana_pred, sample["banana_gt"]),
        "banana_dice": dice_score(banana_pred, sample["banana_gt"]),
        "blemish_iou": iou_score(blemish_pred, sample["blemish_gt"]),
        "blemish_dice": dice_score(blemish_pred, sample["blemish_gt"]),
        "ground_truth_blemish_pct": gt_pct,
        "predicted_blemish_pct": pred_pct,
        "blemish_percentage_error": abs(pred_pct - gt_pct),
    })

final_results = pd.DataFrame(rows)
final_results.to_csv(RESULTS_DIR / "final_evaluation_per_image.csv", index=False)
display(final_results)
print("Saved final_evaluation_per_image.csv")


,image,banana_iou,banana_dice,blemish_iou,blemish_dice,ground_truth_blemish_pct,predicted_blemish_pct,blemish_percentage_error
0,musa-acuminata-banana-8e3c133d-394a-11ec-bc37-...,0.885436,0.939237,0.469552,0.639041,40.541400,36.091307,4.450093
1,musa-acuminata-banana-8f7de794-394a-11ec-a8cc-...,0.864546,0.927353,0.440224,0.611327,39.844137,23.602658,16.241479
2,musa-acuminata-banana-afe6ec6d-394a-11ec-bcab-...,0.976443,0.988081,0.005991,0.011910,3.907305,0.022918,3.884387
3,musa-acuminata-banana-b01b3700-394a-11ec-b4a6-...,0.975709,0.987705,0.297297,0.458333,5.732035,1.685068,4.046968
4,musa-acuminata-banana-b01d2f7c-394a-11ec-a466-...,0.974754,0.987216,0.343779,0.511660,8.439791,3.562369,4.877422
5,musa-acuminata-banana-b2a5cf68-394a-11ec-b7c7-...,0.988356,0.994144,0.334750,0.501592,8.059516,3.052358,5.007159
6,musa-acuminata-freshunripe-1da23724-2653-11ec-...,0.902149,0.948558,0.640836,0.781109,2.812789,2.096703,0.716087
7,musa-acuminata-freshunripe-1fd8dd9a-2653-11ec-...,0.970623,0.985092,0.394578,0.565874,2.666086,6.039175,3.373089
8,musa-acuminata-freshunripe-2cdf79be-2653-11ec-...,0.842397,0.914458,0.000000,0.000000,0.000000,13.711450,13.711450


Saved final_evaluation_per_image.csv


## 3. Final metric summary


In [7]:
final_summary = pd.DataFrame([{
    "images": int(len(final_results)),
    "banana_mean_iou": float(final_results["banana_iou"].mean()),
    "banana_mean_dice": float(final_results["banana_dice"].mean()),
    "blemish_mean_iou": float(final_results["blemish_iou"].mean()),
    "blemish_mean_dice": float(final_results["blemish_dice"].mean()),
    "blemish_percentage_mae": float(final_results["blemish_percentage_error"].mean()),
}])
final_summary.to_csv(RESULTS_DIR / "final_evaluation_summary.csv", index=False)
display(final_summary)
print("Saved final_evaluation_summary.csv")


,images,banana_mean_iou,banana_mean_dice,blemish_mean_iou,blemish_mean_dice,blemish_percentage_mae
0,9,0.931157,0.963538,0.325223,0.453427,6.256459


Saved final_evaluation_summary.csv


## End of Notebook
These are reporting outputs only. The notebook does not interpret whether the result is good enough and does not modify any parameters.
